# 🎙️ GPT-SoVITS 声音克隆 — 听书club 定制版 v3（Drive 持久化）

> 在 Google Colab 免费 GPU (T4) 上完成**声音克隆 + 语音合成**。中文效果最好的开源方案。
>
> **v2 核心改进**：所有中间文件与模型文件挂载到 **Google Drive**，断网/关闭/重开 **不丢失**。
>
> 适用场景：克隆听书club 主持人声线 → 批量生成读书音频。
>
> ⚠️ 需科学上网访问 Colab。免费 T4 有每日时长限制（约12~24h）。
>
> 流程：`挂载 Drive → 准备音频 → 环境配置(一次) → 启动 WebUI → 零样本克隆 或 完整音频微调训练`

## 第 0 步：准备音频（两种模式）

### 模式A — 快速零样本克隆（10 秒即可，先体验效果）
- 3~10 秒**干净人声**（无 BGM/无杂音/单人/中文普通话）
- 用现有语料截取：`ffmpeg -y -ss 0 -t 15 -i "《某书》.mp3" -acodec pcm_s16le -ar 22050 -ac 1 ref_15s.wav`

### 模式B — 完整音频微调训练（推荐！效果最佳）
- **整本音频（30~50 分钟）直接作为训练语料**，WebUI 会自动切分成数百个 3~10 秒片段并 ASR 标注
- 听书club 现有 120+ 本完整音频都可直接用
- 上传方式：把完整 mp3 放到 **Drive 的 corpus 目录**（`MyDrive/GPT-SoVITS/corpus/`），或直接拖到左侧文件面板后移动到该目录
- 数据越多音色越稳，但训练时间也越长（建议单次用 1~2 本，500~1000 步）

> **注意**：所有语料放 Drive 而非 Colab 本地，否则断连丢失。

In [ ]:
# 检查 GPU（必须为 True，且显存 ≥ 15GB 才适合训练；零样本推理 T4 完全够）
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 第 1 步：挂载 Google Drive（持久化核心，每次会话都要跑）

> 授权一次后，Colab 会记住授权（同一 Google 账号）。
> 所有**预训练模型 / 训练产物 / 切片ASR中间文件 / 训练日志**都存在 Drive，断连重开不丢。

In [ ]:
# 挂载 Google Drive（弹出授权 → 选账号 → 粘贴授权码）
from google.colab import drive
import os

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
    print("✅ Drive 已挂载")
else:
    print("✅ Drive 已挂载（跳过授权）")

# 创建持久化工作区（幂等）—— Drive 是唯一真相源，以下目录实际都存这里
WS = "/content/drive/MyDrive/GPT-SoVITS"
for sub in ["pretrained_models", "SoVITS_weights", "GPT_weights", "SoVITS_weights_v2Pro", "GPT_weights_v2Pro", "output", "logs", "corpus"]:
    os.makedirs(f"{WS}/{sub}", exist_ok=True)

print("持久化目录：", WS)
!ls -la "/content/drive/MyDrive/GPT-SoVITS/" 2>/dev/null || true

## 📁 路径约定（重要！两个目录用途不同，别混）

| 路径 | 用途 | 持久化 |
|------|------|--------|
| `/content/GPT-SoVITS/` | GPT-SoVITS **代码仓库** + 运行时临时文件（tmp/ 等） | ❌ 会话结束清空，可重新 clone |
| `/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/` | 预训练模型（**软链**，实际在 Drive） | ✅ |
| `/content/GPT-SoVITS/{logs,output,SoVITS_weights,GPT_weights,SoVITS_weights_v2Pro,GPT_weights_v2Pro}/` | 数据集/切片/特征/训练权重（**软链**，实际在 Drive） | ✅ |
| `/content/drive/MyDrive/GPT-SoVITS/` | **唯一真相源**：上面软链的真实落盘位置 | ✅ 永久 |

> **规则**
> 1. 代码只活在 `/content/GPT-SoVITS`（每次会话 setup.sh 重建或恢复）
> 2. 数据/中间产物/权重一律经软链路径访问（如 `/content/GPT-SoVITS/logs/tingshu_club`），实际写入 Drive
> 3. 训练脚本内部用相对路径（如 `GPT_SoVITS/pretrained_models/...`）时，基准目录是 `/content/GPT-SoVITS`（脚本 cd 到那里执行）
> 4. 工具脚本（slice_audio.py / fasterwhisper_asr.py 等）直接 `from tools.xxx import`，依赖 `PYTHONPATH=/content/GPT-SoVITS`（已写进 env_path.sh + 各 Step cell 内联兜底）；报 `ModuleNotFoundError: No module named 'tools'` 就是 PYTHONPATH 没生效
> 5. 看到 `FileNotFoundError: .../GPT-SoVITS/logs/...` 这类错，先 `ls -la /content/GPT-SoVITS/` 确认软链是否建立（setup.sh 没跑成功 / Drive 没挂载）
> 6. 临时 config 写 `/content/GPT-SoVITS/tmp/`（本地，重启丢但可重建，无需持久化）

## 第 2 步：环境配置（安装/自动恢复，一次性）

> 安装 Anaconda + GPT-SoVITS 代码 + 依赖。
> **首次**：完整安装约 15~25 分钟，安装完自动把 conda 环境打包存 Drive（`MyDrive/GPT-SoVITS/envs/`）。
> **之后重启/硬重置**：跑同一个 cell，自动检测 Drive 里的环境包 → 解包恢复（约 2~3 分钟），**不用重新安装**。
> **预训练模型也存 Drive**，恢复时直接复用、不再下载。
> 日常使用：软重连只跑第 1 步 + 第 3 步；硬重置跑 1→2→3（第 2 步自动走恢复分支，分钟级）。

In [ ]:
# 安装 condacolab（Colab 内的 Anaconda 环境管理，仅首次需要）
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")

# 注意：上面会自动重启运行时，重启后请手动回到这里继续执行下一格

In [ ]:
# 环境配置：自动判断「恢复」或「首次安装」
# - Drive 已有打包好的 conda 环境 → 解包恢复（2~3 分钟）
# - 没有 → 完整安装（15~25 分钟）→ 自动打包存 Drive
%%writefile /content/setup.sh
set -e
WS="/content/drive/MyDrive/GPT-SoVITS"
ENV_TAR="$WS/envs/gptsovits_env.tar.gz"
ENV_DIR="/content/envs/GPTSoVITS"
mkdir -p "$WS/envs"

echo "=== [0/5] 修复 Anaconda 污染的库（libtinfo/libncurses 版本冲突） ==="
# 原因: condacolab 安装 Anaconda 时把旧版库放进 /usr/local/lib，bash 等系统程序
# 启动时会加载到这些旧库，报 "no version information available"
fix_lib() {
  local name="$1"
  if [ -e "/usr/local/lib/$name" ]; then
    local syslib
    syslib=$(find /lib /usr/lib -name "$name*" 2>/dev/null | grep -v /usr/local | head -1)
    if [ -n "$syslib" ]; then
      rm -f "/usr/local/lib/$name"
      ln -s "$syslib" "/usr/local/lib/$name"
      echo "fixed: /usr/local/lib/$name -> $syslib"
    fi
  fi
}
fix_lib libtinfo.so.6
fix_lib libncurses.so.6
fix_lib libncursesw.so.6
fix_lib libreadline.so.8
ldconfig 2>/dev/null || true
echo "=== 库修复完成 ==="

echo "=== [1/5] 代码 ==="
cd /content
if [ ! -d /content/GPT-SoVITS/.git ]; then
  git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS.git
fi
cd /content/GPT-SoVITS

echo "=== [2/5] Drive 软链（训练产物/中间文件/预训练模型持久化） ==="
mkdir -p "$WS/pretrained_models" "$WS/SoVITS_weights" "$WS/GPT_weights" "$WS/SoVITS_weights_v2Pro" "$WS/GPT_weights_v2Pro" "$WS/output" "$WS/logs"
# v2Pro 训练权重目录必须一起软链（s2/s1 的 save_weight_dir 是相对路径，不软链会落本地、重启丢失）
for d in SoVITS_weights GPT_weights SoVITS_weights_v2Pro GPT_weights_v2Pro output logs; do
  if [ ! -L "$d" ]; then
    rm -rf "$d"
    ln -s "$WS/$d" "$d"
    echo "linked: $d -> Drive"
  fi
done
if [ ! -L GPT_SoVITS/pretrained_models ]; then
  rm -rf GPT_SoVITS/pretrained_models
  ln -s "$WS/pretrained_models" GPT_SoVITS/pretrained_models
fi

if [ -f "$ENV_TAR" ]; then
  echo "=== [3/5] 检测到 Drive 环境包 → 恢复（约 2~3 分钟） ==="
  rm -rf "$ENV_DIR"
  mkdir -p "$ENV_DIR"
  tar xzf "$ENV_TAR" -C "$ENV_DIR"
  echo "export PATH=$ENV_DIR/bin:\$PATH" > /content/env_path.sh
  # 工具脚本（slice_audio.py / fasterwhisper_asr.py 等）直接 `from tools.xxx import`，
  # 依赖 /content/GPT-SoVITS 在 sys.path；独立子进程必须靠 PYTHONPATH（webui 进程内 sys.path 自愈但子进程不会继承）
  echo "export PYTHONPATH=/content/GPT-SoVITS" >> /content/env_path.sh
  echo "=== ✅ 恢复完成，跳过安装 ==="
else
  echo "=== [3/5] 首次安装 conda 环境 ==="
  # Colab 默认 conda 源（defaults）里混有 GraalPy 版 python 包，必须强制 conda-forge 拿标准 CPython
  CREATE_CMD="conda create -n GPTSoVITS python=3.10 -y -c conda-forge --override-channels"
  # 健康检查: 若已有 GPTSoVITS env 但不是标准 CPython（如 GraalPy/损坏），删除重建
  if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    # 注意: 环境激活后 conda env list 输出为 "GPTSoVITS * /path"，用 $NF 取最后一列路径
    ENV_PY=$(conda env list | grep -E '^GPTSoVITS' | awk '{print $NF}')
    if [ -n "$ENV_PY" ] && [ -x "$ENV_PY/bin/python" ]; then
      IMPL=$("$ENV_PY/bin/python" -c "import platform; print(platform.python_implementation())" 2>/dev/null || echo "BAD")
      echo "检测到 GPTSoVITS env python 实现: $IMPL"
      if [ "$IMPL" != "CPython" ]; then
        echo "⚠️ 非 CPython（GraalPy 或损坏），删除重建..."
        conda env remove -n GPTSoVITS -y 2>/dev/null || true
        rm -rf "$ENV_PY"   # 彻底删除（conda remove 可能残留）
        $CREATE_CMD
      else
        echo "✅ env 健康 (CPython)"
      fi
    else
      echo "⚠️ env 路径异常，删除重建..."
      conda env remove -n GPTSoVITS -y 2>/dev/null || true
      rm -rf "$ENV_PY"
      $CREATE_CMD
    fi
  else
    $CREATE_CMD
  fi
  source activate GPTSoVITS
  # 激活后再验证一次必须是 CPython（检测 graalpy 关键字兜底）
  IMPL2=$(python -c "import platform, sys; print(platform.python_implementation(), sys.version)" 2>/dev/null || echo "BAD")
  case "$IMPL2" in
    CPython*) echo "✅ python: $IMPL2" ;;
    *)
      echo "❌ 激活后 python 异常: $IMPL2，环境未正确重建，请重试（必要时手动 rm -rf /usr/local/envs/GPTSoVITS）"
      exit 1
      ;;
  esac
  pip install ipykernel -q

  echo "=== [4/5] 安装依赖 + 预训练模型（模型进 Drive） ==="
  # 修复: install.sh 内部用裸 pip 命令（run_pip_quiet 内 `pip install`）
  # 强制替换为 python -m pip（用 env 内 CPython），绕开 PATH 里 GraalPy pip
  ENV_BIN=$(conda env list | grep -E '^GPTSoVITS' | awk '{print $NF}')/bin
  echo "ENV_BIN=$ENV_BIN"
  export PATH="$ENV_BIN:$PATH"
  # sed 替换 install.sh 里所有裸 pip 调用
  sed -i 's/pip install/python -m pip install/g' install.sh
  # 同时创建 pip 包装兜底（PATH 里若有 GraalPy 全局 pip 也拦掉）
  cat > "$ENV_BIN/pip" << 'PIPEOF'
#!/bin/bash
exec "$(dirname "$0")/python" -m pip "$@"
PIPEOF
  chmod +x "$ENV_BIN/pip"
  # 验证 pip 现在指向 CPython
  echo "pip 位置: $(which pip)"
  echo "pip 版本: $(pip --version 2>&1 | head -1)"
  IMPL3=$(python -c "import platform; print(platform.python_implementation())")
  echo "python 实现: $IMPL3"
  if [ "$IMPL3" != "CPython" ]; then
    echo "❌ pip/python 仍非 CPython，中止"
    exit 1
  fi
  bash install.sh --device CU126 --source HF --download-uvr5

  echo "=== [5/5] 打包环境存 Drive（下次恢复用） ==="
  source activate GPTSoVITS
  pip install -q conda-pack
  conda pack -n GPTSoVITS -o /content/gptsovits_env.tar.gz
  cp /content/gptsovits_env.tar.gz "$ENV_TAR"
  # 也解包一份到统一路径，保证 PATH 一致
  mkdir -p "$ENV_DIR"
  tar xzf "$ENV_TAR" -C "$ENV_DIR"
  echo "export PATH=$ENV_DIR/bin:\$PATH" > /content/env_path.sh
  # 工具脚本（slice_audio.py / fasterwhisper_asr.py 等）直接 `from tools.xxx import`，
  # 依赖 /content/GPT-SoVITS 在 sys.path；独立子进程必须靠 PYTHONPATH（webui 进程内 sys.path 自愈但子进程不会继承）
  echo "export PYTHONPATH=/content/GPT-SoVITS" >> /content/env_path.sh
  echo "=== ✅ 首次安装完成，环境已打包存 Drive ==="
fi
echo "=== SETUP DONE ==="


In [ ]:
# 执行环境配置（上面已写入 /content/setup.sh，这里真正运行）
# 首次：完整安装 15~25 分钟；之后重启：自动检测 Drive 环境包 → 2~3 分钟恢复
# 注意：%%writefile 会把整格内容写文件，所以执行必须放独立 cell（否则永远不会运行）
!cd /content && bash setup.sh

## 第 3 步：启动 WebUI（每次使用都要运行）

> 启动后输出一个 **gradio.live 公网链接**，点击打开图形界面。

In [ ]:
# 后台启动 WebUI（约 1~3 分钟），自动抓取公网链接
import subprocess, time, re

for p in ["webui.py", "api.py", "api_v2.py"]:
    subprocess.run(["pkill", "-f", p], capture_output=True)

log_path = "/content/webui.log"
with open(log_path, "w") as f:
    proc = subprocess.Popen(
        ["bash", "-lc", "cd /content/GPT-SoVITS && source /content/env_path.sh && export is_share=True && python webui.py"],
        stdout=f, stderr=subprocess.STDOUT
    )

url = None
for _ in range(180):
    time.sleep(2)
    try:
        log = open(log_path, encoding="utf-8", errors="ignore").read()
    except FileNotFoundError:
        continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", log)
    if m:
        url = m.group(0); break
    if "Traceback" in log and "Error" in log:
        print("⚠️ 启动报错，查看最后 30 行日志：")
        print("\n".join(log.strip().splitlines()[-30:]))
        break

if url:
    print("✅ WebUI 已启动！公网链接（请打开）：")
    print("\n" + "="*60)
    print(url)
    print("="*60)
    print("\n若链接打不开，等 10 秒后执行：!tail -50 /content/webui.log 查看新链接")
else:
    print("⏳ 链接尚未出现，查看日志：")
    print("\n".join(open(log_path, encoding="utf-8", errors="ignore").read().strip().splitlines()[-20:]))

## 第 4 步：使用指南

### A. 零样本克隆（最快）
1. WebUI 标签页 **1-GPT-SoVITS-TTS → 1C-推理**
2. 上传参考音频（模式A的 10 秒片段），填写参考音频的**文字内容**（必须与音频一致）
3. 输入要合成的文本 → 「合成语音」→ 试听/下载

### B. 完整音频微调训练（推荐，听书club 主路径）
> 用**整本音频**训练专属模型，音色最像、最稳。中间文件全部自动落 Drive，断连不丢。

1. **语料就位**：确认完整音频在 `MyDrive/GPT-SoVITS/corpus/`（第 1 步创建）
2. 标签页进入 **1-GPT-SoVITS-TTS → 1A-训练集格式化工具**
   - 音频输入目录：填 `/content/drive/MyDrive/GPT-SoVITS/corpus`
   - 输出目录：填 `output/xxx_opt`（默认 `output/slicer_opt`，实际落在 Drive）
   - 勾选「开启自动 ASR 标注」，语言选中文，模型选 fast whisper large-v3
   - 点击「开启处理」→ 自动切分 + ASR 标注，整本 30-50min 音频约 10~20 分钟
   - 处理完成后输出 `.list` 文件路径（记下来）
3. 标签页进入 **1B-微调训练**
   - 实验名填 `tingshu_club`（可多次用不同名，各自独立）
   - 训练数据路径填上一步的输出 `.list`
   - 依次点击「① 解析数据」→「② 训练 Sovits」→「③ 训练 GPT」
   - 训练 500~1000 步即可（T4 约 30~60 分钟），步数越高越像但也易过拟合
4. 回到 **1C-推理**，模型路径选 `tingshu_club_*`，即可用专属音色合成

> 训练中断不怕：`.list`、切片、特征、权重全在 Drive，重开后直接继续/重训。

## 第 5 步（推荐）：一键全自动训练（无需 WebUI 手动操作）

> 上面第 4 步是 WebUI 图形界面方式。如果你更想要**命令行全自动**：
> 放好完整音频到 corpus → 跑下面 5 格 → 自动完成 切分→ASR→格式化→训练 → 模型直接落 Drive。
> 训练完成后用第 6 步 API 或 WebUI 1C 推理。
>
> 🔌 **防断连三件套**（训练前必做）：
> 1. 先跑下面「防断连保活」格（notebook 内 keepalive 线程）
> 2. 浏览器按 F12 → Console → 粘贴防断连 JS（见本步说明格）
> 3. 保持 Colab 标签页在前台、电脑不锁屏不休眠
>
> 即使断了也不怕：每个 Step 产物都在 Drive，重连后从断掉的 Step 重跑即可，已完成步骤自动跳过。

In [ ]:
# 【防断连保活】训练期间保持连接活跃（先跑本格，再跑 Step1-4）
# 原理：每 60s 在前端刷新一次输出，制造"活跃"信号，防止空闲超时断连
import threading, time, IPython.display

KEEPALIVE_STOP = threading.Event()

def keepalive():
    n = 0
    while not KEEPALIVE_STOP.is_set():
        n += 1
        try:
            IPython.display.clear_output(wait=True)
            print(f"🔄 连接保活中… 已持续 {n} 分钟（每 60s 刷新一次）")
        except Exception:
            pass
        time.sleep(60)

th = threading.Thread(target=keepalive, daemon=True)
th.start()
print("✅ 防断连保活已启动（后台线程，每 60s 刷新输出）")
print("提示：保持 Colab 标签页在前台；电脑设置不休眠；最好在浏览器按 F12 再粘贴防断连 JS（见下格说明）")

## 🔌 浏览器级防断连 JS（最有效，推荐配合使用）

> 在浏览器标签页按 **F12** → 切到 **Console** 标签 → 粘贴下面代码 → 回车。
> 脚本每 60s 自动点击 Colab 的 Connect 按钮，从**浏览器层面**阻止空闲断连（比 notebook 内保活更可靠）。

```javascript
// Colab 防断连：粘贴到浏览器 F12 Console 后回车
function ClickConnect(){
  try {
    var btn = document.querySelector('colab-connect-button') ||
              [...document.querySelectorAll('paper-button')].find(function(b){
                return b.textContent && b.textContent.indexOf('Connect') >= 0;
              });
    if (btn) btn.click();
    console.log('keep-alive tick', new Date().toLocaleTimeString());
  } catch(e) { console.log('keep-alive err', e); }
}
setInterval(ClickConnect, 60000);
// 若报错请换用：document.querySelector('colab-connect-button').click()
```

> 效果验证：Console 里每 60s 出现一次 `keep-alive tick` 即生效。
> ⚠️ 若 Colab 界面改版导致选择器失效，日志会打印 keep-alive err，把新版按钮选择器发给我更新脚本。

In [ ]:
# 【自动训练 Step1】切分完整音频为 3-10s 片段（输出落 Drive output/slicer_opt）
import os, subprocess

WS = "/content/drive/MyDrive/GPT-SoVITS"
CORPUS = f"{WS}/corpus"
SLICE_OUT = f"{WS}/output/slicer_opt"
os.makedirs(SLICE_OUT, exist_ok=True)

# 前置检查：环境是否已恢复（env_path.sh 存在）
if not os.path.exists("/content/env_path.sh") or not os.path.exists("/content/GPT-SoVITS/tools/slice_audio.py"):
    print("❌ 环境未就绪！请先执行第 2 步（setup.sh）完成环境配置/恢复，再跑本格")
else:
    # 列出语料
    corpus_files = [f for f in os.listdir(CORPUS) if f.lower().endswith((".mp3", ".wav", ".m4a", ".flac"))]
    print("语料文件：", corpus_files)

    if not corpus_files:
        print("❌ corpus 目录为空！请先把完整音频上传到 MyDrive/GPT-SoVITS/corpus/")
    else:
        # 幂等：若切片已存在且非空，跳过（断线重跑不浪费）
        existing = [f for f in os.listdir(SLICE_OUT) if f.endswith('.wav')]
        if existing:
            print(f"⏭️ 检测到已有 {len(existing)} 个切片，跳过切分（如想重新切分请清空 {SLICE_OUT}）")
        else:
            # slice_audio.py 参数: inp opt_root threshold min_length min_interval hop_size max_sil_kept _max alpha i_part all_part
            # 参数与官方 WebUI 默认一致: -34 4000 300 10 500 0.9 0.25
            # 注意: 用 pipefail 保证 tail 管道不掩盖真实退出码
            cmd = ["bash", "-lc",
                   f"set -o pipefail; cd /content/GPT-SoVITS && source /content/env_path.sh && export PYTHONPATH=/content/GPT-SoVITS && "
                   f"python tools/slice_audio.py {CORPUS} {SLICE_OUT} -34 4000 300 10 500 0.9 0.25 0 1 2>&1 | tail -5; echo EXIT_CODE=$?"]
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
            out = (r.stdout or "") + (r.stderr or "")
            # 提取 EXIT_CODE（最后一个）
            import re as _re
            m = _re.findall(r"EXIT_CODE=(\d+)", out)
            exit_code = int(m[-1]) if m else 1
            print(out[-1200:])
            if exit_code != 0:
                print(f"❌ slice_audio 退出码 {exit_code}（上面应有报错）")
                print("提示：常见原因是环境未恢复（先跑第2步setup.sh）或语料格式异常")
        n = len([f for f in os.listdir(SLICE_OUT) if f.endswith('.wav')])
        print(f"✅ 切片就绪：{n} 个（Drive output/slicer_opt）")

In [ ]:
# 【自动训练 Step2】ASR 自动标注（Whisper 转写每段文本，输出 .list 到 Drive）
import os, subprocess

WS = "/content/drive/MyDrive/GPT-SoVITS"
SLICE_OUT = f"{WS}/output/slicer_opt"     # 切片输入（Drive）
ASR_OUT = f"{WS}/output/asr_opt"          # 标注输出（Drive）
os.makedirs(ASR_OUT, exist_ok=True)

def run_asr():
    cmd = ["bash", "-lc",
           f"set -o pipefail; cd /content/GPT-SoVITS && source /content/env_path.sh && export PYTHONPATH=/content/GPT-SoVITS && "
           f"python tools/asr/fasterwhisper_asr.py -i {SLICE_OUT} -o {ASR_OUT} -s large-v3 -l zh -p int8 2>&1 | tail -10; echo EXIT_CODE=$?"]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    out = (r.stdout or "") + (r.stderr or "")
    import re as _re
    m = _re.findall(r"EXIT_CODE=(\d+)", out)
    exit_code = int(m[-1]) if m else 1
    print(out[-1200:])
    return exit_code

# 幂等：.list 已存在则跳过（断线重跑不浪费）；否则只跑一次
lists = [f for f in os.listdir(ASR_OUT) if f.endswith(".list")]
if lists:
    list_path = os.path.join(ASR_OUT, lists[0])
    with open(list_path, encoding="utf-8", errors="ignore") as f:
        content = f.read()
    cnt = len(content.strip().split("\n")) if content.strip() else 0
    print(f"⏭️ 检测到已有标注文件 {lists[0]}（{cnt} 条），跳过 ASR")
else:
    print("=== 运行 faster-whisper ASR（large-v3，中文）===")
    exit_code = run_asr()
    if exit_code != 0:
        print(f"❌ ASR 退出码 {exit_code}（上面应有报错）")
    lists = [f for f in os.listdir(ASR_OUT) if f.endswith(".list")]
    list_path = os.path.join(ASR_OUT, lists[0]) if lists else None

print("生成标注文件：", lists)
if list_path:
    # 显示前几行供校对
    with open(list_path, encoding="utf-8", errors="ignore") as f:
        lines = f.read().strip().split("\n")[:3]
    print("标注样例（前3条）：")
    for ln in lines: print(" ", ln[:120])
    # 保存路径供 Step3 使用
    with open("/content/asr_list_path.txt", "w") as f:
        f.write(list_path)
    print("\n✅ ASR 就绪 →", list_path)
else:
    print("❌ ASR 未生成 .list，先解决上面的报错再跑 Step3")

In [ ]:
# 【自动训练 Step3】格式化数据集（v2Pro 全量：文本→HuBERT→声纹→语义 + 分片合并）
# 耗时：5~10min（已生成的步骤自动跳过）。产物落 Drive logs/tingshu_club/
# 路径约定：/content/GPT-SoVITS/logs 是软链 → 实际写入 /content/drive/MyDrive/GPT-SoVITS/logs
import os, subprocess, re, glob

WS = "/content/drive/MyDrive/GPT-SoVITS"
EXP_NAME = "tingshu_club"
EXP_ROOT = "/content/GPT-SoVITS/logs"     # 软链到 Drive 的 logs
S1_DIR = f"{EXP_ROOT}/{EXP_NAME}"          # 数据集目录（实际在 Drive）
os.makedirs(S1_DIR, exist_ok=True)

try:
    list_path = open("/content/asr_list_path.txt").read().strip()
except Exception:
    list_path = os.path.join(WS, "output/asr_opt", "slicer_opt.list")
assert os.path.exists(list_path), f"❌ 找不到 ASR 标注文件: {list_path}，请先跑 Step2"

def run_train(cmd, timeout=3600):
    r = subprocess.run(["bash", "-lc",
                        f"set -o pipefail; cd /content/GPT-SoVITS && source /content/env_path.sh && {cmd} 2>&1 | tail -30; echo EXIT_CODE=$?"],
                       capture_output=True, text=True, timeout=timeout)
    out = (r.stdout or "") + (r.stderr or "")
    m = re.findall(r"EXIT_CODE=(\d+)", out)
    exit_code = int(m[-1]) if m else 1
    print(out[-1500:])
    if exit_code != 0:
        print(f"❌ 退出码 {exit_code}（上面应有报错）")
    return exit_code

def missing(name):
    return not os.path.exists(f"{S1_DIR}/{name}")

# 新版 GPT-SoVITS prepare_datasets 脚本必需的 env（旧版没这些会静默失败/直接报错）
env = {
    "inp_text": list_path,
    "inp_wav_dir": f"{WS}/output/slicer_opt",          # 切片输入（Drive）
    "exp_name": EXP_NAME,
    "opt_dir": S1_DIR,                                   # 数据集输出（Drive，经软链）
    "bert_pretrained_dir": "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large",
    "cnhubert_base_dir": "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/chinese-hubert-base",
    "pretrained_s2G": "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/v2Pro/s2Gv2Pro.pth",
    "s2config_path": "/content/GPT-SoVITS/GPT_SoVITS/configs/s2v2Pro.json",
    "sv_path": "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/sv/pretrained_eres2netv2w24s4ep4.ckpt",
    "i_part": "0", "all_parts": "1", "is_half": "True",
}
envs = " ".join(f'{k}="{v}"' for k, v in env.items())

# 幂等：缺什么跑什么（断线重跑不浪费）
steps = []
if missing("2-name2text.txt") or missing("3-bert"):
    steps.append(("1) 文本分词 (1-get-text)", "1-get-text.py"))
if missing("4-cnhubert") or missing("5-wav32k"):
    steps.append(("2) HuBERT 特征 (2-get-hubert-wav32k)", "2-get-hubert-wav32k.py"))
if missing("7-sv_cn"):
    steps.append(("3) 声纹特征 (2-get-sv，v2Pro 必需)", "2-get-sv.py"))
if missing("6-name2semantic.tsv"):
    steps.append(("4) 语义 token (3-get-semantic)", "3-get-semantic.py"))

for name, script in steps:
    print(f"=== {name} ===")
    rc = run_train(f"env {envs} python -s GPT_SoVITS/prepare_datasets/{script}", timeout=3600)
    if rc != 0:
        print(f"⚠️ {name} 失败"); raise SystemExit

# 合并分片（新版脚本按 i_part 分片输出，官方 webui 事后合并成单文件）
# 文本: 2-name2text-0.txt → 2-name2text.txt（s1 训练读）
if missing("2-name2text.txt"):
    parts = sorted(glob.glob(f"{S1_DIR}/2-name2text-*.txt"))
    if parts:
        opt = []
        for p in parts:
            with open(p, encoding="utf8") as f:
                opt += f.read().strip("\n").split("\n")
            os.remove(p)
        with open(f"{S1_DIR}/2-name2text.txt", "w", encoding="utf8") as f:
            f.write("\n".join(opt) + "\n")
        print(f"✅ 合并 2-name2text.txt：{len(opt)} 行")
    else:
        print("⚠️ 没有 2-name2text-*.txt 分片"); raise SystemExit

# 语义: 6-name2semantic-0.tsv → 6-name2semantic.tsv（带表头，s1 训练读）
if missing("6-name2semantic.tsv"):
    parts = sorted(glob.glob(f"{S1_DIR}/6-name2semantic-*.tsv"))
    if parts:
        opt = ["item_name\tsemantic_audio"]
        for p in parts:
            with open(p, encoding="utf8") as f:
                opt += f.read().strip("\n").split("\n")
            os.remove(p)
        with open(f"{S1_DIR}/6-name2semantic.tsv", "w", encoding="utf8") as f:
            f.write("\n".join(opt) + "\n")
        print(f"✅ 合并 6-name2semantic.tsv：{len(opt)-1} 行")
    else:
        print("⚠️ 没有 6-name2semantic-*.tsv 分片"); raise SystemExit

# 最终检查：5 项全绿才允许进入训练
print("== 数据集最终检查 ==")
ok = True
for f in ["2-name2text.txt", "4-cnhubert", "5-wav32k", "7-sv_cn", "6-name2semantic.tsv"]:
    exists = os.path.exists(f"{S1_DIR}/{f}")
    ok &= exists
    print(f"  {f}: {'✅' if exists else '❌ 缺失'}")
sz = os.path.getsize(f"{S1_DIR}/6-name2semantic.tsv") if os.path.exists(f"{S1_DIR}/6-name2semantic.tsv") else 0
if not ok or sz < 31:
    print("⚠️ 数据集不完整，先解决上面的问题再跑 Step4"); raise SystemExit
print(f"✅ 数据集就绪（语义文件 {sz}B，Drive logs/tingshu_club/），可以跑 Step4 训练")

In [ ]:
# 【自动训练 Step4】正式训练（SoVITS 声学模型 → GPT 语义模型）
# 官方新版训练必须用 config 文件驱动（CLI 直传参数已废弃），这里完整复刻 WebUI 逻辑
# 权重目录 SoVITS_weights_v2Pro / GPT_weights_v2Pro 已软链到 Drive，训练直接落盘、重启不丢
import os, subprocess, json, yaml

WS = "/content/drive/MyDrive/GPT-SoVITS"
EXP_NAME = "tingshu_club"
EXP_ROOT = "/content/GPT-SoVITS/logs"     # 软链到 Drive 的 logs
S1_DIR = f"{EXP_ROOT}/{EXP_NAME}"          # 数据集目录（Drive）
TOTAL_EPOCH = 10        # 或改小：5（更快，音色稍差）；调大：15（更像，更慢）
BATCH_SIZE = 4          # T4 16GB 建议 4；OOM 改 2
VERSION = "v2Pro"       # 官方默认版本（v2Pro 最稳，中文效果最佳）
TMP = "/content/GPT-SoVITS/tmp"   # 本地临时 config（重启丢，可重建，无需持久化）
os.makedirs(TMP, exist_ok=True)

def run_train(cmd, timeout=10800):
    r = subprocess.run(["bash", "-lc",
                        f"set -o pipefail; cd /content/GPT-SoVITS && source /content/env_path.sh && {cmd} 2>&1 | tail -30; echo EXIT_CODE=$?"],
                       capture_output=True, text=True, timeout=timeout)
    out = (r.stdout or "") + (r.stderr or "")
    import re as _re
    m = _re.findall(r"EXIT_CODE=(\d+)", out)
    exit_code = int(m[-1]) if m else 1
    print(out[-1500:])
    if exit_code != 0:
        print(f"❌ 退出码 {exit_code}（上面应有报错）")
    return exit_code

semantic = f"{S1_DIR}/6-name2semantic.tsv"
phoneme  = f"{S1_DIR}/2-name2text.txt"
for f in [semantic, phoneme]:
    assert os.path.exists(f), f"❌ 缺少数据集文件: {f}，请先跑 Step3 格式化"

# ---------- SoVITS (s2) 训练：生成 tmp_s2.json（复刻 webui open1Ba，字段全量补齐） ----------
sovits_dir = f"{WS}/SoVITS_weights_v2Pro"    # 软链目标 = Drive 真实目录
sovits_done = os.path.exists(sovits_dir) and any(f.endswith('.pth') for f in os.listdir(sovits_dir))
if sovits_done:
    print("⏭️ SoVITS 权重已存在，跳过 s2 训练")
else:
    print("=== 生成 s2 config + 训练 SoVITS 声学模型 ===")
    with open("/content/GPT-SoVITS/GPT_SoVITS/configs/s2v2Pro.json") as f:
        data = json.load(f)
    os.makedirs(f"{S1_DIR}/logs_s2_{VERSION}", exist_ok=True)
    data["train"]["batch_size"] = BATCH_SIZE
    data["train"]["epochs"] = TOTAL_EPOCH
    data["train"]["text_low_lr_rate"] = 1.0
    data["train"]["pretrained_s2G"] = f"GPT_SoVITS/pretrained_models/v2Pro/s2G{VERSION}.pth"
    data["train"]["pretrained_s2D"] = f"GPT_SoVITS/pretrained_models/v2Pro/s2D{VERSION}.pth"
    data["train"]["if_save_latest"] = True
    data["train"]["if_save_every_weights"] = False
    data["train"]["save_every_epoch"] = 5
    data["train"]["gpu_numbers"] = "0"              # 单卡 T4（缺这个 s2_train 直接 AttributeError）
    data["train"]["grad_ckpt"] = False
    data["train"]["lora_rank"] = 32
    data["model"]["version"] = VERSION
    data["data"]["exp_dir"] = data["s2_ckpt_dir"] = S1_DIR
    data["save_weight_dir"] = "SoVITS_weights_v2Pro"   # 已软链 → 实际写入 Drive
    data["name"] = EXP_NAME
    data["version"] = VERSION
    with open(f"{TMP}/tmp_s2.json", "w") as f:
        f.write(json.dumps(data))
    rc = run_train(f"python -s GPT_SoVITS/s2_train.py --config {TMP}/tmp_s2.json")
    if rc != 0:
        print("⚠️ s2 训练失败，检查上方日志（常见：预训练模型路径不对/显存不足）")

# ---------- GPT (s1) 训练：生成 tmp_s1.yaml（复刻 webui open1Bb） ----------
gpt_dir = f"{WS}/GPT_weights_v2Pro"        # 软链目标 = Drive 真实目录
gpt_done = os.path.exists(gpt_dir) and any(f.endswith('.ckpt') for f in os.listdir(gpt_dir))
if gpt_done:
    print("⏭️ GPT 权重已存在，跳过 s1 训练")
else:
    print("\n=== 生成 s1 config + 训练 GPT 语义模型 ===")
    with open("/content/GPT-SoVITS/GPT_SoVITS/configs/s1longer-v2.yaml") as f:
        data = yaml.safe_load(f)
    os.makedirs(f"{S1_DIR}/logs_s1_{VERSION}", exist_ok=True)
    data["train"]["batch_size"] = BATCH_SIZE
    data["train"]["epochs"] = TOTAL_EPOCH
    data["train"]["save_every_n_epoch"] = 1
    data["train"]["if_save_every_weights"] = True
    data["train"]["if_save_latest"] = True
    data["train"]["half_weights_save_dir"] = "GPT_weights_v2Pro"   # 已软链 → 实际写入 Drive
    data["train"]["exp_name"] = EXP_NAME
    data["pretrained_s1"] = "GPT_SoVITS/pretrained_models/s1v3.ckpt"
    data["train_semantic_path"] = semantic
    data["train_phoneme_path"] = phoneme
    data["output_dir"] = f"{S1_DIR}/logs_s1_{VERSION}"
    with open(f"{TMP}/tmp_s1.yaml", "w") as f:
        yaml.dump(data, f, default_flow_style=False)
    rc2 = run_train(f"python -s GPT_SoVITS/s1_train.py --config_file {TMP}/tmp_s1.yaml")
    if rc2 != 0:
        print("⚠️ s1 训练失败，检查上方日志")

print("\n✅ 训练流程结束！权重经软链已落 Drive：")
print("  GPT_weights_v2Pro/  →", os.listdir(gpt_dir) if os.path.exists(gpt_dir) else "（无，检查日志）")
print("  SoVITS_weights_v2Pro/ →", os.listdir(sovits_dir) if os.path.exists(sovits_dir) else "（无，检查日志）")

## 第 6 步（可选）：API 批量合成 + 长文本分段

启动 OpenAI 兼容 API（端口 9880），配合脚本实现「长文本分段 → 批量合成 → ffmpeg 合并」，接入听书club 生产流水线。训练好的模型也存 Drive，API 直接调用。

In [ ]:
# 启动 API 服务（v2，OpenAI 兼容，端口 9880）
import subprocess, time, socket

api_log = "/content/api.log"
with open(api_log, "w") as f:
    proc = subprocess.Popen(
        ["bash", "-lc",
         "cd /content/GPT-SoVITS && source /content/env_path.sh && "
         "python api_v2.py -a 127.0.0.1 -p 9880 -c GPT_SoVITS/configs/tts_infer.yaml 2>&1"],
        stdout=f, stderr=subprocess.STDOUT
    )

ready = False
for _ in range(120):
    time.sleep(2)
    s = socket.socket(); s.settimeout(1)
    if s.connect_ex(("127.0.0.1", 9880)) == 0:
        ready = True; s.close(); break
    s.close()

if ready:
    print("✅ API 服务已就绪 (127.0.0.1:9880)，支持 OpenAI /v1/audio/speech 接口")
else:
    print("⚠️ API 未就绪，日志：")
    print(open(api_log, encoding="utf-8", errors="ignore").read()[-1000:])

In [ ]:
# API 调用示例（零样本克隆：传参考音频 + 文本 → 返回合成音频）
import requests, base64

# 1) 上传参考音频获得 reference_id（只需做一次）
#    参考音频建议放在 Drive，断连不丢：/content/drive/MyDrive/GPT-SoVITS/corpus/ref_15s.wav
ref_audio = "/content/drive/MyDrive/GPT-SoVITS/corpus/ref_15s.wav"
ref_text  = "大家好，歡迎來到《听书club》，今天為你解讀的是"  # 改成参考音频的实际文字

with open(ref_audio, "rb") as f:
    base64_audio = base64.b64encode(f.read()).decode()

resp = requests.post("http://127.0.0.1:9880/change_refer", json={
    "refer_wav_path": ref_audio,
    "prompt_text": ref_text,
    "prompt_language": "zh"
}, timeout=60)
print("change_refer:", resp.json())

# 2) 合成文本（OpenAI 兼容格式）
text = "大家好，歡迎來到《听书club》。今天為你解讀的是《反脆弱》：那些杀不死我们的，终将使我们更强大。"
r = requests.post("http://127.0.0.1:9880/v1/audio/speech", json={
    "model": "GPT-SoVITS",
    "input": text,
    "voice": "default"
}, timeout=120)
print("HTTP:", r.status_code, "bytes:", len(r.content))
if r.status_code == 200:
    with open("/content/output_api.wav", "wb") as f:
        f.write(r.content)
    print("✅ 已保存 /content/output_api.wav （可在左侧文件面板下载，或拷回 Drive 保存）")

## 长文本批量合成（听书club 10,000 字脚本）

```python
import re, subprocess, requests

text = open("/content/drive/MyDrive/GPT-SoVITS/corpus/script.txt", encoding="utf-8").read()
# 按句切分，每段 ≤ 300 字（保证合成稳定）
chunks, cur = [], ""
for s in re.split(r"(?<=[。！？\n])", text):
    if len(cur) + len(s) > 300: chunks.append(cur); cur = s
    else: cur += s
if cur: chunks.append(cur)

parts = []
for i, ch in enumerate(chunks):
    r = requests.post("http://127.0.0.1:9880/v1/audio/speech",
                      json={"model": "GPT-SoVITS", "input": ch, "voice": "default"}, timeout=180)
    if r.status_code == 200:
        p = f"/content/part_{i:03d}.wav"; open(p, "wb").write(r.content); parts.append(p)
        print(i, "OK", len(ch), "字")

# 合并（输出到 Drive，断连不丢）
with open("/content/concat.txt", "w") as f:
    for p in parts: f.write(f"file '{p}'\n")
subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", "/content/concat.txt",
                "-c", "copy", "/content/drive/MyDrive/GPT-SoVITS/corpus/final_clone.wav"])
print("✅ 完成：" + str(len(parts)) + " 段，已存 Drive corpus/final_clone.wav")
```

## ⚠️ 断线恢复 & 常见问题

### 断线/重开恢复流程（核心！）
| 情况 | 处理 |
|------|------|
| **软重连**（Colab 顶部重连，磁盘保留） | 跑第 1 步（挂 Drive）→ 跑第 3 步（启动 WebUI），环境还在，秒级恢复 |
| **硬重置**（关闭太久/换机器，/content 清空） | 跑 1→2→3：第 2 步**自动检测 Drive 里的环境包并解包恢复**（2~3 分钟），无需重新安装；预训练模型/中间文件/语料全在 Drive，直接继续训练/推理 |
| **训练中断** | 切片/ASR/.list/特征/权重都在 Drive，重新打开后从断点继续 |
| **第5步自动训练中断** | 每个 Step 有幂等检查：重连后从断掉的 Step 重跑即可，已完成步骤自动跳过（切片存在跳过Step1、.list存在跳过Step2、Step3 按「缺什么跑什么」细粒度跳过、v2Pro 权重存在跳过Step4） |
| **FileNotFoundError（找不到 logs/tingshu_club/xxx）** | 先 `ls -la /content/GPT-SoVITS/` 看软链是否建立（logs/weights/output 应显示 → Drive）。没软链 = setup.sh 没跑成功或 Drive 没挂载；有软链 = 对应步骤还没生成该产物，重跑 Step3 |

### 其他
- **参考音频文字必须准确**：零样本克隆相似度 80% 取决于参考音频与文字一致性，用 Whisper 转写后校对
- **模型位置**：训练产物在 `MyDrive/GPT-SoVITS/GPT_weights_v2Pro/` 和 `SoVITS_weights_v2Pro/`（经软链直接落盘），永远在 Drive，无需手动备份
- **Drive 空间**：免费 15GB。一份训练约 1-2GB，预训练模型约 2GB；装不下时在 Drive 清理旧实验
- **速度**：训练时中间文件在 Drive，比本地盘略慢（可接受）；若明显拖慢，把 `logs/` 软链改回本地 `rm logs && mkdir logs`，训练完拷回 Drive
- **声音不像**：换更干净的参考音频、增加微调步数、参考音频用同性别同语速
- **显存不足**：T4 16GB 训练 1000 步没问题；报 OOM 就换新实验名重训（历史权重占显存）

---
*Notebook 由 Hermes Agent 生成 · v3 修复新版 GPT-SoVITS 格式化/训练 · Drive 持久化 · 基于 RVC-Boss/GPT-SoVITS 官方 Colab-WebUI.ipynb*